# Project_R — Rapido supply working

Open in Colab (this file) → **Runtime → Run all**. It clones the repo if the CSVs are not already next to the notebook, runs the pipeline, and renders a Rapido-yellow dashboard.

Local: same **Run all**. CSVs already in the repo root.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KRRamanathan/Project_R/blob/cursor/working-notebook-82ee/Project_R.ipynb)


In [ ]:
# ── 0. Workspace (Google Colab + local) ──────────────────────────────────────
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("PROJECT_R_REPO", "https://github.com/KRRamanathan/Project_R.git")
BRANCH = os.environ.get("PROJECT_R_BRANCH", "cursor/working-notebook-82ee")


def _has_data(p: Path) -> bool:
    return (p / "captains.csv").exists() and (p / "metrics.py").exists()


def _pip(pkgs: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])


cwd = Path.cwd().resolve()
if _has_data(cwd):
    ROOT = cwd
else:
    dest = Path("/content/Project_R") if IN_COLAB else (cwd / "_project_r_clone")
    if not _has_data(dest):
        if dest.exists():
            shutil.rmtree(dest)
        print(f"Cloning {REPO_URL} @ {BRANCH} → {dest}")
        subprocess.check_call(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(dest)]
        )
    os.chdir(dest)
    ROOT = dest.resolve()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

if IN_COLAB:
    _pip(
        [
            "pandas>=2.0",
            "numpy>=1.24",
            "scipy>=1.11",
            "statsmodels>=0.14",
            "matplotlib>=3.8",
            "python-docx>=1.1",
            "python-pptx>=0.6.23",
            "ipywidgets>=8.0",
        ]
    )
else:
    req = ROOT / "requirements.txt"
    if req.exists():
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

print("IN_COLAB", IN_COLAB)
print("ROOT", ROOT)
print("CSVs", sorted(p.name for p in ROOT.glob("*.csv")))


In [ ]:
# ── 1. Theme ────────────────────────────────────────────────────────────────
from IPython.display import display, HTML, Image, clear_output
import notebook_app as ui

ui.inject_css()
ui.hero()


In [ ]:
# ── 2. Run the working (quiet + progress chips; logs stay folded) ───────────
import contextlib
import io
import runpy
import time

STEPS = [
    ("01_data_audit.py", "Audit"),
    ("02_funnel.py", "Funnel"),
    ("03_dropoff.py", "Drop-off"),
    ("04_channel_leaks.py", "Channel"),
    ("05_campaign.py", "Campaign"),
    ("06_airport_hourly.py", "Airport hours"),
    ("07_airport_trips.py", "Airport trips"),
    ("08_intervention_sizing.py", "Sizing"),
    ("09_deliverables.py", "Packet"),
]

logs: dict[str, str] = {}
done: list[str] = []

for script, label in STEPS:
    display(HTML(ui.progress_html(done, current=label)))
    buf = io.StringIO()
    t0 = time.time()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        runpy.run_path(str(ROOT / script), run_name="__main__")
    logs[label] = buf.getvalue()
    done.append(f"{label} {time.time()-t0:.0f}s")
    clear_output(wait=True)
    ui.inject_css()
    ui.hero()
    display(HTML(ui.progress_html(done)))

display(HTML('<div class="pr-wrap"><div class="pr-ok"><b>Pipeline finished.</b> Same maths as ./run_all.sh · memo + deck written if Step 9 ran.</div></div>'))


In [ ]:
# ── 3. Dashboard ────────────────────────────────────────────────────────────
import pandas as pd
from metrics import (
    C1A_CENTRAL_SHOWUP,
    airport_hourly_snapshot,
    ara_economics,
    ara_monthly_cost,
    c1a_monthly,
    c1b_monthly,
    event_funnel,
    headlines,
    load_onboarding,
    mature_months,
)
import check_regression

h = headlines()
eco = ara_economics()
hourly = airport_hourly_snapshot()
months, span_days, *_ = mature_months()
funnel = event_funnel()
n = len(funnel)
n_appr = int(funnel["final_status"].eq("approved").sum())
captains, *_ = load_onboarding()
mix = captains["vehicle_type"].value_counts().to_dict()

ui.metric_cards(
    [
        ("Approved | mature∩events", f"{100 * n_appr / n:.1f}%", f"{n_appr:,} / {n:,} · {months:.2f} mo window"),
        ("C1a extra / month", f"{c1a_monthly(C1A_CENTRAL_SHOWUP):.0f}", "60% show-up × RC att-3 · unobserved"),
        ("C1b extra / month", f"{c1b_monthly(2):.0f}", "Insurance capture UX @ att-2"),
        ("C1a + C1b (disjoint)", f"{c1a_monthly(C1A_CENTRAL_SHOWUP) + c1b_monthly(2):.0f}", "Do not add fos 128–256"),
        ("Airport unfulfilled", f"{100 * hourly['airport_unf_share']:.0f}%", f"vs {100 * hourly['other_unf_share']:.0f}% elsewhere · 84% is 21:00–03:59"),
        ("ARA / eligible leg", f"₹{eco['payout_per_eligible_leg']:.1f}", f"sample ₹{ara_monthly_cost(1):,.0f}/mo · not city P&L"),
    ]
)
ui.vehicle_strip(mix, "Auto · Cab · ERickshaw in file · Bike not in extract")
ui.rec_cards()


In [ ]:
# ── 4. Live assumptions (smooth sliders) ────────────────────────────────────
def _panel(show_up: int, ara_pct: int):
    c1a = c1a_monthly(show_up / 100.0)
    c1b = c1b_monthly(2)
    ara = ara_monthly_cost(ara_pct / 100.0)
    pay = eco["payout_per_eligible_leg"] * ara_pct / 100.0
    ui.metric_cards(
        [
            ("C1a show-up", f"{show_up}%", "unobserved · central case 60%"),
            ("C1a / month", f"{c1a:.1f}", "flow × show-up × RC att-3"),
            ("Onboarding C1a+C1b", f"{c1a + c1b:.1f} / mo", "Insurance C1b stays put"),
            ("ARA payout", f"₹{pay:.1f}/leg", f"{ara_pct}% of derived ₹35.4"),
            ("ARA sample / month", f"₹{ara:,.0f}", "sampled trips · not a budget line"),
        ]
    )

try:
    import ipywidgets as w
    from IPython.display import display as _d
    _d(_panel)
    w.interact(
        _panel,
        show_up=w.IntSlider(
            value=60, min=0, max=100, step=1, description="C1a show-up %",
            style={"description_width": "130px"}, layout=w.Layout(width="70%"),
        ),
        ara_pct=w.IntSlider(
            value=100, min=50, max=150, step=5, description="ARA % of ₹35.4",
            style={"description_width": "130px"}, layout=w.Layout(width="70%"),
        ),
    )
except Exception as exc:
    print("Sliders unavailable (", type(exc).__name__, ") — showing 60% / 100%.")
    _panel(60, 100)


In [ ]:
# ── 5. Waterfall + regression ───────────────────────────────────────────────
wf = ROOT / "figures" / "funnel_waterfall.png"
if wf.exists():
    display(HTML('<div class="pr-wrap"><div class="pr-sec">Funnel waterfall · mature ∩ events</div></div>'))
    display(Image(filename=str(wf)))
else:
    display(HTML('<div class="pr-warn">Waterfall missing — Step 9 did not write figures/funnel_waterfall.png</div>'))

print("Regression lock")
rc = check_regression.main()
if rc != 0:
    raise SystemExit("Headline regression failed")


In [ ]:
# ── 6. Step logs (folded) ───────────────────────────────────────────────────
display(HTML('<div class="pr-wrap"><div class="pr-sec">Full step prints</div><p style="color:#5C5C5C">Same text as 01_…09_*_output.txt. Collapsed so the dashboard stays readable.</p></div>'))
for _script, label in STEPS:
    ui.log_block(label, logs.get(label, ""))
